# Colombia Open — RETHUS (talento humano en salud)

Este notebook muestra cómo cargar el dataset **`rethus`** (Registro Único Nacional del Talento Humano en Salud: conteo de registros de talento humano en salud por perfil profesional, programa académico, origen del título y año de registro, por municipio de residencia — MinSalud) con `colombia-open`.

> **Snapshot congelado, no serie viva.** La fuente declara frecuencia de actualización anual, pero no se ha refrescado desde 2022-05-30 — trata este dataset como una foto del talento humano en salud registrado hasta esa fecha, no como el estado actual del registro. Ver `docs/datasets/rethus.md` para el detalle.

> **Nota:** `colombia-open` está publicado en [PyPI](https://pypi.org/project/colombia-open/) — `pip install colombia-open` descarga y cachea el bundle de datos publicado en GitHub Releases automáticamente.

## Setup

In [ ]:
!pip install -q colombia-open

## Cargar el dataset

In [ ]:
import polars as pl
from colombia_open import ColombiaOpen

hub = ColombiaOpen()
rethus = hub.load_polars("rethus")

print(rethus.shape)
rethus.head(10)

## Esquema

| Columna | Tipo | Descripción |
|---|---|---|
| `cod_dpto` / `cod_mpio` | string (2 / 5 dígitos) | Departamento/municipio de **residencia** del profesional — join key hacia `municipios`. |
| `nombre_dpto` / `nombre_mpio` | string | Nombres parseados del mismo campo de la fuente. |
| `origen_titulo` | string | "1 - LOCAL", "2 - EXTRANJERO" o "NO DEFINIDO". |
| `tipo_institucion` | string | "1 - EDUCACIÓN SUPERIOR" o "2 - EDUCACIÓN PARA EL TRABAJO Y EL DESARROLLO HUMANO". |
| `tipo_programa` | string | AUXILIAR, TÉCNICO PROFESIONAL, TECNOLOGÍA, UNIVERSITARIA, ESPECIALIZACIÓN, MAESTRÍA, DOCTORADO o NO DEFINIDO. |
| `perfil_profesional` | string | Perfil con su código embebido (ej. "P07 - MEDICINA", "A02 - AUXILIAR EN ENFERMERÍA"). 156 valores. |
| `anio_registro` | int | Año del acto administrativo de registro. |
| `numero_registros` | int | Conteo de registros para esa combinación exacta — súmalo para totales, nunca cuentes filas. |

Esquema completo: [`docs/datasets/rethus.md`](../../docs/datasets/rethus.md).

## Total de talento humano en salud por municipio (top 10)

In [ ]:
(
    rethus.group_by(["cod_mpio", "nombre_mpio"])
    .agg(pl.col("numero_registros").sum().alias("total"))
    .sort("total", descending=True)
    .head(10)
)

## Perfiles profesionales con más registros (nacional)

In [ ]:
(
    rethus.group_by("perfil_profesional")
    .agg(pl.col("numero_registros").sum().alias("total"))
    .sort("total", descending=True)
    .head(10)
)

## Por qué no ves años recientes: el snapshot está congelado

El año más reciente con datos reales es 2022 (parcial — la fuente se congeló el 2022-05-30). Cualquier año posterior a 2022 que aparezca es un defecto de captura conocido de la fuente, no un dato real nuevo.

In [ ]:
rethus.group_by("anio_registro").agg(pl.col("numero_registros").sum().alias("total")).sort(
    "anio_registro", descending=True
).head(10)

## Limitaciones conocidas

- **Snapshot congelado a 2022-05, no una serie viva** — no uses este dataset para inferir tendencias recientes sin verificar primero si MinSalud publicó un recurso más actualizado.
- **Es residencia del profesional, no el lugar donde ejerce.**
- **No es microdato** — cada fila ya es un conteo agregado por la fuente.
- **2076 filas de la fuente cruda se excluyeron** por traer "SIN INFORMACION" en departamento y municipio de residencia a la vez.

Documentación completa: [`docs/datasets/rethus.md`](../../docs/datasets/rethus.md).